**Environment Installation**


In [ ]:

!pip install datasets transformers langchain-text-splitters sentence-transformers faiss-cpu google-genai

**Imports and Global Setups**

Import all required libraries and securely load the API Key.

In [ ]:
import os
import time
import faiss
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google import genai
from google.colab import userdata

# Securely load the Gemini API key
try:
    api_key = userdata.get('Gemini_Api_Key')
except userdata.SecretNotFoundError:
    raise ValueError("GEMINI_API_KEY not found in Colab Secrets. Please add it via the left sidebar.")

# 2. Inject it into the environment variables so the SDK can find it
os.environ["Gemini_Api_Key"] = api_key

# 3. Initialize the client
client = genai.Client(api_key=os.environ["Gemini_Api_Key"])
print("Gemini API Client successfully initialized.")

# 4. Test the connection
models = list(client.models.list())
print("Total models found:", len(models))

**Data Ingestion and Deduplication**

We extract the SciQ dataset and remove the duplicates in order to preserve its exact choronological order.

In [ ]:
print("Downloading SciQ dataset...")
dataset = load_dataset("allenai/sciq")

unique_supports_dict = {}
all_splits = ["train", "validation", "test"]

for split in all_splits:
    for ex in dataset[split]:
        s = ex["support"]
        if s and isinstance(s, str) and s.strip():
            unique_supports_dict[s.strip()] = True

# Convert keys back to a list to preserve original dataset order
ordered_unique_supports = list(unique_supports_dict.keys())
print(f"Total unique source paragraphs extracted for the library: {len(ordered_unique_supports)}")
print(f"First 5 unique source paragraphs extracted for the library: {ordered_unique_supports[:5]}")

**Pipeline functions**

These are the functions we built to handle the specific experimental variables.

In [ ]:
def build_chunked_corpus(unique_paragraphs, chunk_size, model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer=tokenizer,
        chunk_size=chunk_size,
        chunk_overlap=25,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    documents = []
    for i, paragraph in enumerate(unique_paragraphs):
        chunks = text_splitter.split_text(paragraph)
        for chunk in chunks:
            documents.append({"id": len(documents), "text": chunk})
    return documents

def build_vector_index(documents, model_name):
    model = SentenceTransformer(model_name)
    texts = [doc["text"] for doc in documents]

    embeddings = model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
    faiss.normalize_L2(embeddings)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension) # Cosine similarity
    index.add(embeddings)

    return index, model

**Generative QA and Retrieval Logic**

This cell contains the logic to search the FAISS index and strictly prompt the LLM to generate an answer based only on the context.

In [ ]:
def retrieve_context(query, index, model, documents, k):
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    _, idxs = index.search(q_emb, k)
    return [documents[i]["text"] for i in idxs[0]]

def evaluate_rag_config(split_name, index, embed_model, documents, k, n_samples):
    data = dataset[split_name].select(range(min(n_samples, len(dataset[split_name]))))
    correct, total = 0, 0

    for i, ex in enumerate(data):
        q = ex["question"]
        gold_answer = ex["correct_answer"].lower()

        # 1. Retrieve
        ctx_passages = retrieve_context(q, index, embed_model, documents, k)
        context_str = "\n\n".join([f"[Context {j+1}]\n{p}" for j, p in enumerate(ctx_passages)])

        # 2. Format Prompt
        prompt = (
            "You are a strict evaluation assistant taking an open-book test.\n"
            "You must answer the question based STRICTLY AND ONLY on the provided Context.\n"
            "If the answer is NOT explicitly written in the Context, you MUST output exactly: NOT_FOUND.\n"
            "Do not use your outside knowledge. Keep your answer under 10 words.\n\n"
            f"Context:\n{context_str}\n\n"
            f"Question: {q}\n\n"
            "Answer:"
        )

        # 3. Generate Answer (using an older model to avoid the 2.5-flash daily limit)
        try:
            response = client.models.generate_content(
                #model='gemini-3-flash-preview',
                # model='gemini-1.5-flash-8b',
               # model='gemini-2.5-flash-lite',
               model= 'models/gemini-2.5-flash',


                contents=prompt,
            )
            pred_text = response.text.strip().lower()
        except Exception as e:
            print(f"API Error on question {i}: {e}")
            pred_text = ""
            time.sleep(10) # Wait out the rate limit

        # 4. Strict Substring Evaluation
        if gold_answer in pred_text:
            correct += 1

        total += 1

        # RATE LIMIT PRECAUTION (15 seconds between calls)
        time.sleep(5)

# Before running the loop, do a quick test:
    try:
        test_resp = model.generate_content("test")
        print("API Connection Successful")
    except Exception as e:
        print(f"API Connection Failed: {e}")

    return correct / total if total > 0 else 0




In [ ]:
# [CELL 6]
# --- YOUR EXPERIMENTAL VARIABLES ---
# 1. Domain Specificity
EMBED_MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2", # General Baseline
    "pritamdeka/S-PubMedBert-MS-MARCO"        # Domain-Specific (Medical/Science)
]

# 2. Granularity
CHUNK_SIZES = [128, 256, 512]

# 3. Volume
TOP_K_VALUES = [1, 3, 5]

# How many questions to test per configuration (Keep this tiny until API limits are lifted)
N_SAMPLES = 20

print("=== STARTING RAG ABLATION STUDY ===")


# The Master Loop
for model_name in EMBED_MODELS:
    print(f"\n{'='*50}\nEVALUATING MODEL: {model_name}\n{'='*50}")

    for chunk_size in CHUNK_SIZES:
        print(f"\n--- Building Infrastructure | Chunk Size: {chunk_size} ---")

        # Build Corpus & Index for this specific configuration
        current_docs = build_chunked_corpus(ordered_unique_supports, chunk_size, model_name)
        current_index, current_model = build_vector_index(current_docs, model_name)

        for k in TOP_K_VALUES:
            acc = evaluate_rag_config(
                split_name="validation",
                index=current_index,
                embed_model=current_model,
                documents=current_docs,
                k=k,
                n_samples=N_SAMPLES
            )
            print(f"  [Result] Model: {model_name.split('/')[-1]} | Chunk: {chunk_size} | Top-k: {k} | Accuracy: {acc:.2%}")

In [ ]:
# Create a new cell and run this code
import os
from google import genai
from google.colab import userdata

# Initialize client
api_key = userdata.get('Gemini_Api_Key')
client = genai.Client(api_key=api_key)

print("=== AVAILABLE TEXT GENERATION MODELS ===")
for m in client.models.list():
    # We only care about models that can generate text/content
    if "generateContent" in m.supported_actions:
        print(m.name)